# Ablation Study

Measures the contribution of each training component **on the real external test
set** (`evals/ext_eval/revizied_evals.xlsx` — 100 manually de-identified radiology
reports, the same benchmark as `baselines.ipynb`). One shared loader (§1) feeds
every row and the decoding ablation (§6). Runs **top-to-bottom in one pass**:
models are loaded in-process (transformers), no vLLM server required.

| Row | Pseudo-label | GRPO | QC | Model | How produced |
|---|---|---|---|---|---|
| A | — | — | — | `Qwen/Qwen3-0.6B-Base` | inference |
| B | ✓ | ✗ | ✗ | `EdgeDeID/Qwen3-0.6B-Base_SFT` | inference |
| C | ✓ | ✗ | ✓ | — | **B + QC** (derived) |
| D | ✓ | ✓ | ✗ | `EdgeDeID/Qwen3-0.6B-GRPO` | inference |
| E | ✓ | ✓ | ✓ | — | **D + QC** (derived) |

Rows A/B/D are generated by the in-process loop in §2 (one checkpoint at a time,
GPU freed between). Rows C/E are derived by applying the QC pipeline to B/D — no
extra inference. Predictions cache to `ablation/`. §6 adds the greedy-vs-union
decoding ablation on the GRPO checkpoint.

## Colab setup

Run this first **only if you're on Colab / a remote kernel**. It clones the repo, installs `edgedeid`, and prompts for your GitHub + HF tokens. On a local run it's a no-op.

In [2]:
# ==== Colab / remote setup — run me FIRST (no-op when not on Colab) ====
# Clones the repo (branch `refactor`), installs the package, and collects tokens
# via getpass so nothing secret is saved into the notebook. On a local run this
# detects "not Colab" and does nothing.
def _on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _on_colab():
    import os, getpass
    REPO = "/content/EdgeDeID"
    if not os.path.exists(os.path.join(REPO, "pyproject.toml")):
        _gh = getpass.getpass("GitHub token (repo scope): ")
        !git clone -q --branch refactor https://{_gh}@github.com/BardiaKh/EdgeDeID.git {REPO}
        _gh = None
    %cd {REPO}
    %pip install -q -e .
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (private EdgeDeID/* checkpoints): ")
    print("Colab setup done — now run the repo-bootstrap cell below.")
else:
    print("Not on Colab — skipping remote setup (using local repo).")

/content/EdgeDeID
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for edgedeid (pyproject.toml) ... done
Colab setup done — now run the repo-bootstrap cell below.


In [1]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    # installed editable (pip install -e .)? locate the repo via the package itself
    # — robust on Colab / Kaggle / remote kernels where cwd isn't inside the repo.
    try:
        import edgedeid, pathlib
        _cand = pathlib.Path(edgedeid.__file__).resolve().parent.parent
        if (_cand / "pyproject.toml").exists():
            ROOT = str(_cand)
    except Exception:
        pass
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = ("/content/EdgeDeID" if os.path.exists("/content/EdgeDeID/pyproject.toml")
            else "/kaggle/input/datasets/sinadaneshgar/edgedeid")  # last-resort fallbacks
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/sinadaneshgar/edgedeid'

## 0 — Setup

In [3]:
import os

import pandas as pd
from edgedeid.eval import compute_metrics, print_report

ABLATION_DIR = "ablation"          # per-row prediction cache
os.makedirs(ABLATION_DIR, exist_ok=True)

# Ordered rows: (key, label, served_model_or_None, derive_from_or_None).
# A/C/E are produced by inference against a served model; D/F are derived by
# applying QC to another row's predictions.
ABLATION_ROWS = [
    ("A", "Base (0-shot)",          "Qwen/Qwen3-0.6B-Base",          None),
    ("B", "SFT (pseudo-label)",     "EdgeDeID/Qwen3-0.6B-Base_SFT",  None),
    ("C", "SFT + QC",               None,                            "B"),
    ("D", "SFT + GRPO",             "EdgeDeID/Qwen3-0.6B-GRPO",      None),
    ("E", "SFT + GRPO + QC (full)", None,                            "D"),
]
ROW_LABEL   = {k: lab for k, lab, _, _ in ABLATION_ROWS}
ROW_MODEL   = {k: m   for k, _, m, _ in ABLATION_ROWS}
ROW_DERIVE  = {k: d   for k, _, _, d in ABLATION_ROWS}


def row_csv(key):
    return os.path.join(ABLATION_DIR, f"{key}_predictions.csv")

## 0.5 — Preflight smoke test  (run me first)

Verifies the notebook can run top-to-bottom **before** any expensive inference:
the dataset, core imports, the exact load path, and optional deps. Hard problems
raise immediately; optional gaps print a `warn` with the one-line fix.

In [5]:
import edgedeid, pathlib, os, sys
ROOT = str(pathlib.Path(edgedeid.__file__).resolve().parent.parent)
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT,
      "| xlsx present:", os.path.exists(f"{ROOT}/evals/ext_eval/revizied_evals.xlsx"))


repo root: /content/EdgeDeID | xlsx present: True


In [6]:
# ── Preflight: fail fast, all issues at once ──────────────────────────────────
import importlib, xml.etree.ElementTree as ET

_hard, _warn = [], []
def _ok(m):  print(f"  ok   {m}")
def _bad(m): _hard.append(m); print(f"  FAIL {m}")
def _wn(m):  _warn.append(m); print(f"  warn {m}")

print("[1] dataset + load path")
DATA = f"{ROOT}/evals/ext_eval/revizied_evals.xlsx"
if not os.path.exists(DATA):
    _bad(f"missing dataset: {DATA}")
else:
    try:
        from edgedeid.data.preprocessing import preprocess_text
        from edgedeid.data.xml_utils import convert_to_xml
        _df = pd.read_excel(DATA)
        _miss = {"default_response_0", "default_response_1"} - set(_df.columns)
        if _miss:
            _bad(f"xlsx missing columns: {_miss}")
        else:
            _s = preprocess_text(_df.head(3).copy(),
                                 input_col="default_response_0", output_col="default_response_1")
            for _v in _s["default_response_1"].apply(convert_to_xml):
                ET.fromstring(_v)                      # raises on malformed GT-XML
            _ok(f"{len(_df)} rows · preprocess_text + convert_to_xml valid")
    except Exception as e:
        _bad(f"load path broken: {type(e).__name__}: {e}")

print("[2] core imports")
for _mod, _sym in [("edgedeid.eval","compute_metrics"), ("edgedeid.eval","print_report")]:
    try: getattr(importlib.import_module(_mod), _sym); _ok(f"{_mod}.{_sym}")
    except Exception as e: _bad(f"{_mod}.{_sym}: {e}")

print("[3] inference + QC path (import only, no weights)")
try:
    from edgedeid.baselines import QwenDeidBaseline
    from edgedeid.eval import make_qc_config, union_inproc
    from edgedeid.qc import apply_qc_batch
    make_qc_config(use_tier3=True)                     # constructs a QCConfig
    _ok("QwenDeidBaseline · make_qc_config · union_inproc · apply_qc_batch")
except Exception as e:
    _bad(f"inference/QC import: {e}")

print("[4] optional deps")
try:
    import torch; _ok(f"torch CUDA={torch.cuda.is_available()}")
except Exception as e: _bad(f"torch missing (required for in-process inference): {e}")
def _ensure_spacy(model="en_core_web_sm"):
    """Install spaCy + the model on first run so QC Tier-3 is available."""
    import importlib, subprocess, sys
    try:
        import spacy
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy"], check=True)
        importlib.invalidate_caches(); import spacy
    try:
        spacy.load(model)
    except OSError:
        print(f"  ... downloading spaCy {model} (one-time)")
        subprocess.run([sys.executable, "-m", "spacy", "download", model], check=True)
        importlib.invalidate_caches(); spacy.load(model)
    return True
try:
    _ensure_spacy(); _ok("spaCy en_core_web_sm ready (QC Tier-3)")
except Exception as e:
    _wn(f"spaCy en_core_web_sm unavailable ({type(e).__name__}) → QC Tier-3 degrades to Tier-2")
if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
    _ok("HF token present (private EdgeDeID/* checkpoints)")
else:
    _wn("no HF_TOKEN → rows C/E (private EdgeDeID/*) will 401 (set HF_TOKEN)")

print("\n" + "=" * 60)
if _hard:
    print(f"PREFLIGHT FAILED — {len(_hard)} hard issue(s); fix before running top-to-bottom:")
    for _m in _hard: print("   -", _m)
    raise RuntimeError("preflight hard failures: " + "; ".join(_hard))
print(f"PREFLIGHT PASSED  ({len(_warn)} warning(s) — see 'warn' lines above)")

[1] dataset + load path
Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name               18           15
  age                 3            -
  date                9            3
  place               3            -
  address             2            -
  phone               1            -
  email               1            -
  id                  1            -
  url                 4            -
  device              1            -

Text processing pipeline completed!
  ok   100 rows · preprocess_text + convert_to_xml valid
[2] core imports
  ok   edgedeid.eval.compute_metrics
  ok   edgedeid.eval.print_report
[3] inferen

## 1 — Load the external test set (`revizied_evals.xlsx`)

The single source of truth for every row and the decoding ablation. Identical to
`baselines.ipynb` §1: rule-based `preprocess_text` builds the gold `DeID`, then
`convert_to_xml` builds the gold `<phi_entities>` XML. Produces `notes` and
`gt_xmls`.

In [7]:
from edgedeid.data.preprocessing import preprocess_text
from edgedeid.data.xml_utils import convert_to_xml

# Revised external eval set (xlsx): default_response_0 = raw note,
# default_response_1 = inline PHI-tagged gold. These are preprocess_text's
# native column names, so no renaming is needed.
raw_df = pd.read_excel(f"{ROOT}/evals/ext_eval/revizied_evals.xlsx")
raw_df = preprocess_text(raw_df, input_col="default_response_0", output_col="default_response_1")
raw_df["GT-XML"] = raw_df["default_response_1"].apply(convert_to_xml)

NOTE_COL = "default_response_0"
for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[NOTE_COL] = raw_df[NOTE_COL].str.replace(tag, "", regex=False).str.strip()

notes   = raw_df[NOTE_COL].tolist()
gt_xmls = raw_df["GT-XML"].tolist()
print(f"{len(notes)} evaluation samples (revizied_evals.xlsx)")

Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name              542          449
  age                73            -
  date              271           56
  place              74            -
  address            67            -
  phone              71            -
  email              62            -
  id                 63            -
  url               115            -
  device              6            -

Text processing pipeline completed!
100 evaluation samples (revizied_evals.xlsx)


## 2 — Generate per-row predictions (in-process, one pass)

**Inference rows (A/C/E)** are produced by loading each checkpoint **in-process**
via `QwenDeidBaseline` (transformers) and running **greedy** batched inference on
`notes`. The loop below does all three in one pass — no vLLM server, no manual
model swapping — freeing GPU memory between checkpoints. Each row is cached to
`ablation/<key>_predictions.csv`.

**Derived rows (C/E)** apply the QC pipeline to B and D. QC Tier-3 (spaCy NER)
auto-degrades to Tier-2 if spaCy isn't installed, so this never crashes.

> The two `EdgeDeID/*` checkpoints are private — set `HF_TOKEN` in the
> environment (or `huggingface-cli login`) before running. A checkpoint that
> fails to load is reported and skipped; its row simply shows as missing.

In [8]:
import gc

import torch
from tqdm import tqdm

from edgedeid.baselines import QwenDeidBaseline
from edgedeid.eval import make_qc_config
from edgedeid.qc import apply_qc_batch

USE_TIER3  = True     # include spaCy NER (name/place/address) in C/E QC
BATCH_SIZE = 16       # notes per forward pass for greedy batched inference

# Tier-3 auto-degrades to Tier-2 if spaCy / en_core_web_sm is missing (no crash).
qc_cfg = make_qc_config(use_tier3=USE_TIER3)


def free_model(m):
    """Drop refs and release GPU memory before the next checkpoint loads."""
    for attr in ("model", "tokenizer"):
        if hasattr(m, attr):
            try: setattr(m, attr, None)
            except Exception: pass
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_row(key, preds):
    df = pd.DataFrame({"note": notes, "LLM-XML": preds, "GT-XML": gt_xmls})
    df.to_csv(row_csv(key), index=False)
    print(f"  saved [{key}] {ROW_LABEL[key]} -> {row_csv(key)} ({len(df)} rows)")
    return df

In [9]:
# ── Generate inference rows A/C/E in-process (one checkpoint at a time) ───────
INFERENCE_KEYS = [k for k, _, m, _ in ABLATION_ROWS if m is not None]  # A, B, D

for key in INFERENCE_KEYS:
    model_src = ROW_MODEL[key]
    print(f"\n{'='*70}\n[{key}] {ROW_LABEL[key]}  <-  {model_src}\n{'='*70}")
    try:
        model = QwenDeidBaseline(model_src, name=ROW_LABEL[key])
    except Exception as e:
        import traceback
        print(f"  LOAD FAILED ({type(e).__name__}): {e} — skipping row [{key}]")
        traceback.print_exc()
        continue
    try:
        preds = []
        for i in tqdm(range(0, len(notes), BATCH_SIZE), desc=key):
            preds.extend(model.predict_xml_batch(notes[i:i + BATCH_SIZE]))
        save_row(key, preds)
    finally:
        free_model(model)

print("\nInference rows generated:", [k for k in INFERENCE_KEYS if os.path.exists(row_csv(k))])

A:   0%|          | 0/7 [02:39<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ── DERIVE-QC (rows D and F) ─────────────────────────────────────────────────
# Applies the QC pipeline to the parent row's cached predictions. D = C + QC,
# F = E + QC. Uses the Tier-3-guarded qc_cfg, so it runs whether or not spaCy is
# installed (Tier-2 recall floor always applies).
for key in ("C", "E"):
    parent = ROW_DERIVE[key]
    if not os.path.exists(row_csv(parent)):
        print(f"  [{key}] skipped — parent row [{parent}] not generated")
        continue
    parent_preds = pd.read_csv(row_csv(parent))["LLM-XML"].fillna("").tolist()
    qc_preds, _ = apply_qc_batch(parent_preds, notes, config=qc_cfg)
    save_row(key, qc_preds)
    print(f"  [{key}] = [{parent}] + QC (Tier-3={qc_cfg.use_tier3})")

## 3 — Compute metrics for each generated row

In [ ]:
ablation_data, results = {}, {}
for key, label, _, _ in ABLATION_ROWS:
    path = row_csv(key)
    if not os.path.exists(path):
        print(f"  [{key}] {label}: *** NOT GENERATED — skipping ***")
        continue
    df = pd.read_csv(path)
    preds = df["LLM-XML"].fillna("").tolist()
    gts   = df["GT-XML"].fillna("").tolist()
    nts   = df["note"].tolist() if "note" in df.columns else None
    r = compute_metrics(preds, gts, source_notes=nts)
    ablation_data[key] = (preds, gts, nts)
    results[key] = (label, r)
    print_report(r, title=f"[{key}] {label}")
    print()

print(f"{len(results)} / {len(ABLATION_ROWS)} rows evaluated.")

## 4 — Ablation table

In [ ]:
rows = []
for key, label, _, _ in ABLATION_ROWS:
    if key not in results:
        rows.append({"Row": key, "Config": label, "Micro-P": None, "Micro-R": None,
                     "Micro-F1": None, "F-beta(b=2)": None, "Macro-F1": None,
                     "Zero-leak": None, "XML-Parse": None, "Hallucination": None})
        continue
    _, r = results[key]
    rows.append({
        "Row": key, "Config": label,
        "Micro-P":       round(r.micro_precision, 4),
        "Micro-R":       round(r.micro_recall, 4),
        "Micro-F1":      round(r.micro_f1, 4),
        "F-beta(b=2)":   round(r.f_beta, 4),
        "Macro-F1":      round(r.macro_f1, 4),
        "Zero-leak":     round(r.zero_leak_rate, 4),
        "XML-Parse":     round(r.xml_parse_rate, 4),
        "Hallucination": round(r.hallucination_rate, 4),
    })

ablation_df = pd.DataFrame(rows).set_index("Row")
ablation_df

## 5 — Δ vs previous row (incremental gain)

In [ ]:
numeric_cols = ["Micro-R", "Micro-F1", "F-beta(b=2)", "Zero-leak", "Hallucination"]
delta_df = ablation_df[numeric_cols].apply(pd.to_numeric, errors="coerce").diff().round(4)
delta_df.columns = [f"Δ {c}" for c in delta_df.columns]
combined = pd.concat([ablation_df, delta_df], axis=1)
combined

In [ ]:
os.makedirs("artifacts/ablation", exist_ok=True)
ablation_df.to_csv("artifacts/ablation/ablation_results.csv")
print("Saved artifacts/ablation/ablation_results.csv")

## 6 — Decoding-strategy ablation: greedy + QC vs. K-sample union

Recall here uses **directional containment** (`metrics._covers`: a GT entity is
protected only when a prediction covers its *whole* span). More complete spans
therefore raise recall. Two ways to chase completeness:

- **Greedy (T=0)** — one deterministic forward pass. Completeness comes from
  training (`reward_completeness`, β=2) + deterministic QC injection.
- **K-sample union (T>0)** — sample each note K times, union the detected PHI
  spans. The union can only *grow*, so recall is monotone non-decreasing in K,
  at **K× the inference cost**. It also aggregates a few hallucinated spans
  (precision cost), but for de-id **over-redaction is safe**.

Does cheap deterministic QC beat expensive union sampling on recall / zero-leak?
That is the result that justifies the edge-deployment architecture. Runs
**in-process** on the same checkpoint as row **D** (GRPO) — no vLLM.

| Arm | Decoding | QC | Forward passes / note |
|-----|----------|----|----|
| G   | greedy (T=0)         | — | 1 |
| G+QC| greedy (T=0)         | Tier-2(+3) | 1 |
| U   | union (K draws, T>0) | — | K |
| U+QC| union (K draws, T>0) | Tier-2(+3) | K |

In [ ]:
# ── config ───────────────────────────────────────────────────────────────────
from edgedeid.eval import union_inproc

DECODE_MODEL = ROW_MODEL["D"]   # GRPO checkpoint — the deployed model
K            = 5                 # samples per note for the union arms
TEMP         = 0.7               # sampling temperature for the union arms
SAMPLE_BATCH = 8                 # notes per forward pass while sampling (K× sequences)

assert "notes" in globals() and "gt_xmls" in globals(), \
    "Run §1 first (defines notes, gt_xmls)."
print(f"{len(notes)} notes | decode model = {DECODE_MODEL} | K={K} T={TEMP} "
      f"| QC Tier-3={qc_cfg.use_tier3}")

In [ ]:
# ── run the four arms (in-process on one loaded checkpoint) ───────────────────
decode_model = QwenDeidBaseline(DECODE_MODEL, name="decoding-ablation")
try:
    # G : greedy, no QC            (1 pass/note)
    preds_greedy = union_inproc(decode_model, notes, temperature=0.0,
                                batch_size=BATCH_SIZE)
    # U : K-sample union, no QC    (K passes/note)
    preds_union  = union_inproc(decode_model, notes, k=K, temperature=TEMP,
                                batch_size=SAMPLE_BATCH)
finally:
    free_model(decode_model)

# +QC : deterministic Tier-2(+3) reconciliation on top of each decoding
preds_greedy_qc, _ = apply_qc_batch(preds_greedy, notes, config=qc_cfg)
preds_union_qc, _  = apply_qc_batch(preds_union,  notes, config=qc_cfg)

DECODING_ARMS = [
    ("G  · greedy",              preds_greedy,    1),
    ("G+QC · greedy+QC",         preds_greedy_qc, 1),
    (f"U  · union(K={K})",       preds_union,     K),
    (f"U+QC · union(K={K})+QC",  preds_union_qc,  K),
]
print("Arms ready:", [a for a, _, _ in DECODING_ARMS])

In [ ]:
# ── metrics + cost table ─────────────────────────────────────────────────────
dec_rows = []
for label, preds, cost in DECODING_ARMS:
    r = compute_metrics(preds, gt_xmls, source_notes=notes)
    dec_rows.append({
        "Arm":            label,
        "Micro-P":        round(r.micro_precision, 4),
        "Micro-R":        round(r.micro_recall, 4),
        "Micro-F1":       round(r.micro_f1, 4),
        "F-beta(b=2)":    round(r.f_beta, 4),
        "Zero-leak":      round(r.zero_leak_rate, 4),
        "Halluc.":        round(r.hallucination_rate, 4),
        "Passes/note":    cost,
    })

decoding_df = pd.DataFrame(dec_rows).set_index("Arm")
# Recall / zero-leak gain per extra forward pass — the efficiency argument.
base_r  = decoding_df.loc["G  · greedy", "Micro-R"]
base_zl = decoding_df.loc["G  · greedy", "Zero-leak"]
decoding_df["ΔRecall vs G"]   = (decoding_df["Micro-R"]   - base_r).round(4)
decoding_df["ΔZero-leak vs G"] = (decoding_df["Zero-leak"] - base_zl).round(4)
decoding_df

In [ ]:
# ── export ───────────────────────────────────────────────────────────────────
os.makedirs("artifacts/ablation", exist_ok=True)
decoding_df.to_csv("artifacts/ablation/decoding_ablation.csv")
print("Saved artifacts/ablation/decoding_ablation.csv")

# Headline read for the paper: does greedy+QC match/beat union sampling at 1/K cost?
gqc = decoding_df.loc["G+QC · greedy+QC"]
u   = decoding_df.loc[f"U  · union(K={K})"]
uqc = decoding_df.loc[f"U+QC · union(K={K})+QC"]
print(f"greedy+QC : recall={gqc['Micro-R']:.4f} zero-leak={gqc['Zero-leak']:.4f} @ 1 pass")
print(f"union(K={K}): recall={u['Micro-R']:.4f} zero-leak={u['Zero-leak']:.4f} @ {K} passes")
print(f"union+QC  : recall={uqc['Micro-R']:.4f} zero-leak={uqc['Zero-leak']:.4f} @ {K} passes")
verdict = ("greedy+QC matches/beats union sampling at 1/K the cost"
           if gqc["Micro-R"] >= u["Micro-R"] else
           "union sampling adds recall beyond greedy+QC — quantify the cost trade")
print("VERDICT:", verdict)